# Environment

In [1]:
import os 

from pathlib import Path
from PIL import Image
from transformers import AutoTokenizer, AutoProcessor, AutoModelForImageTextToText



ROOT_DIR = Path("../../").resolve()

model_path = str(ROOT_DIR / "weights" / "Nanonets-OCR-s")   # model_id: "nanonets/Nanonets-OCR-s"
model_id = "nanonets/Nanonets-OCR-s"

/home/octoopt/anaconda3/envs/turlio/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model = AutoModelForImageTextToText.from_pretrained(
    model_id, 
    torch_dtype="auto", 
    device_map="auto", 
    attn_implementation="flash_attention_2"
)
model.eval()

tokenizer = AutoTokenizer.from_pretrained(model_id)
processor = AutoProcessor.from_pretrained(model_id)

Loading checkpoint shards: 100%|██████████| 2/2 [00:02<00:00,  1.22s/it]
Some parameters are on the meta device because they were offloaded to the cpu.
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [3]:
# def ocr_page_with_nanonets_s(img_base64, model, processor, max_new_tokens=4096):
#     prompt = """Extract the text from the above document as if you were reading it naturally. Return the tables in html format. Return the equations in LaTeX representation. If there is an image in the document and image caption is not present, add a small description of the image inside the <img></img> tag; otherwise, add the image caption inside <img></img>. Watermarks should be wrapped in brackets. Ex: <watermark>OFFICIAL COPY</watermark>. Page numbers should be wrapped in brackets. Ex: <page_number>14</page_number> or <page_number>9/22</page_number>. Prefer using ☐ and ☑ for check boxes."""
#     # image = Image.open(image_path)
#     messages = [
#         {"role": "system", "content": "You are a helpful assistant."},
#         {"role": "user", "content": [
#             {"type": "image", "image": f"data:image/png;base64,{img_base64}"},
#             {"type": "text", "text": prompt},
#         ]},
#     ]
#     text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
#     inputs = processor(text=[text], images=[image], padding=True, return_tensors="pt")
#     inputs = inputs.to(model.device)
    
#     output_ids = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
#     generated_ids = [output_ids[len(input_ids):] for input_ids, output_ids in zip(inputs.input_ids, output_ids)]
    
#     output_text = processor.batch_decode(generated_ids, skip_special_tokens=True, clean_up_tokenization_spaces=True)
#     return output_text[0]


import base64
from io import BytesIO
from PIL import Image

def ocr_page_with_nanonets_s(img_base64: str, model, processor, max_new_tokens: int = 4096):
    """
    Run OCR extraction with multimodal LLM (e.g., Nanonets, Florence, IDEFICS).
    Works with base64-encoded image input (PNG/JPG).
    """

    # Decode base64 image
    try:
        # strip prefix if exists (e.g. data:image/png;base64,)
        if "," in img_base64:
            img_base64 = img_base64.split(",", 1)[1]
        image_data = base64.b64decode(img_base64)
        image = Image.open(BytesIO(image_data)).convert("RGB")
    except Exception as e:
        raise ValueError(f"Invalid base64 image data: {e}")

    # Prompt instruction
    prompt = (
        "Extract the text from the above document as if you were reading it naturally. "
        "Return the tables in HTML format. Return the equations in LaTeX representation. "
        "If there is an image in the document and image caption is not present, add a small "
        "description of the image inside the <img></img> tag; otherwise, add the image caption "
        "inside <img></img>. Watermarks should be wrapped in brackets. Ex: <watermark>OFFICIAL COPY</watermark>. "
        "Page numbers should be wrapped in brackets. Ex: <page_number>14</page_number> or <page_number>9/22</page_number>. "
        "Prefer using ☐ and ☑ for check boxes."
    )

    # Build chat template for the vision-language model
    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {
            "role": "user",
            "content": [
                {"type": "image", "image": "data:image/png;base64," + img_base64},
                {"type": "text", "text": prompt},
            ],
        },
    ]

    # Prepare inputs for the model
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[image], padding=True, return_tensors="pt")
    inputs = inputs.to(model.device)

    # Generate output
    output_ids = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(inputs.input_ids, output_ids)
    ]

    output_text = processor.batch_decode(
        generated_ids, skip_special_tokens=True, clean_up_tokenization_spaces=True
    )
    return output_text[0]


In [9]:
from pdf2image import convert_from_path
from io import BytesIO
import base64

def pdf_to_base64_images(pdf_path: str):
    """
    Convert PDF pages to a list of base64-encoded PNG images.
    """
    pages = convert_from_path(pdf_path, dpi=300)
    base64_images = []
    for page in pages:
        buffer = BytesIO()
        page.save(buffer, format="PNG")
        img_base64 = base64.b64encode(buffer.getvalue()).decode("utf-8")
        base64_images.append(img_base64)
    return base64_images


In [11]:
filepath = str(ROOT_DIR / "dataset" / "Docling Technical Report.pdf")


base64_images = pdf_to_base64_images(filepath)

for i, img_base64 in enumerate(base64_images):
    text = ocr_page_with_nanonets_s(img_base64, model, processor)
    print(f"=== Page {i+1} ===")
    print(text)


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


OutOfMemoryError: CUDA out of memory. Tried to allocate 210.00 MiB. GPU 0 has a total capacity of 3.94 GiB of which 15.31 MiB is free. Including non-PyTorch memory, this process has 3.92 GiB memory in use. Of the allocated memory 3.74 GiB is allocated by PyTorch, and 119.29 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [4]:
import base64


filepath = str(ROOT_DIR / "dataset" / "Docling Technical Report.pdf")
# Read the PDF file in binary mode
with open(filepath, "rb") as pdf_file:
    pdf_bytes = pdf_file.read()

# Encode the PDF bytes to base64
base64_string = base64.b64encode(pdf_bytes).decode("utf-8")

In [7]:
filepath

'/home/octoopt/workspace/projects/learn-from-basics/the-notes/dataset/Docling Technical Report.pdf'

In [16]:
result = ocr_page_with_nanonets_s(base64_string, model, processor, max_new_tokens=15000)
print(result)

ValueError: Invalid base64 image data: cannot identify image file <_io.BytesIO object at 0x77b2ef0751c0>

## Model Setup

## Training/Validating Process

## Inference

# Conclusion


...